In [1]:
from stepmix.stepmix import StepMix
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pickle

In [2]:
cluster_df = pd.read_csv("../../data/master_data/2016_to_2023_clustering_input_data.csv")

len(cluster_df)

85659

In [3]:
cluster_cols = ["Age9",
                "Gend3",
                "Eth7",
                "Disab2_POP",
                "Educ6",
                "NSSEC5",
                "IMD10",
                "WorkStat8",
                "Child4",
                "HHLiv9",
                "Motiva_POP",
                "motivd_POP"]

X = cluster_df[cluster_cols]

X.head()

,Age9,Gend3,Eth7,Disab2_POP,Educ6,NSSEC5,IMD10,WorkStat8,Child4,HHLiv9,Motiva_POP,motivd_POP
0,0,0,3,1,0,0,1,1,1,2,0,3
1,2,0,3,1,0,1,3,1,0,1,1,3
2,2,1,0,1,0,0,6,0,0,0,1,3
3,2,0,1,1,0,0,3,0,1,4,1,3
4,2,1,1,0,0,0,3,1,1,4,0,2


In [4]:
results = []

for k in [24, 28]:

    try:
    
        print(f"Fitting {k} classes...")

        model = StepMix(n_components=k,  measurement="categorical", random_state=42, n_init=50, max_iter=5000, abs_tol=1e-5)

        model.fit(X)

        with open(f"stepmix_{k}_class_model.pkl", "wb") as f:
             pickle.dump(model, f)

        posterior = model.predict_proba(X)
        predicted = model.predict(X)

        proportions = posterior.mean(axis=0)
        assignment = posterior.max(axis=1)

        loglik = model.score(X) * len(X)

        print(f"Log-likelihood : {loglik:,.2f}")
        print(f"AIC            : {model.aic(X):,.2f}")
        print(f"BIC            : {model.bic(X):,.2f}")
        print(f"Converged      : {model.converged_}")
        print(f"Iterations     : {model.n_iter_}")

        print("\nClass proportions")
        print(pd.Series(proportions).round(4))

        print("\nObserved class sizes")
        print(pd.Series(predicted).value_counts(normalize=True).sort_index().round(4))

        print("\nPosterior assignment certainty")
        print(pd.Series(assignment).describe())

        print(f"\nMean assignment probability : {assignment.mean():.4f}")
        print(f"Median assignment           : {np.median(assignment):.4f}")
        print(f"Minimum assignment          : {assignment.min():.4f}")

        temp = cluster_df.copy()
        temp["Class"] = predicted

        print("\nYear by latent class")
        print(pd.crosstab(temp["year"], temp["Class"], normalize="index").round(3))

        print("\nModel attributes")
        print(sorted(model.__dict__.keys()))

        params = model.get_parameters()

        print("\nParameter keys:")
        print(params.keys())

        measurement = params["measurement"]

        print("\nMeasurement parameters")
        print(type(measurement))

        if isinstance(measurement, dict):
            print("Measurement keys:")
            print(measurement.keys())

            for key, value in measurement.items():
                print(f"\n{key}")
                print(type(value))
                if hasattr(value, "shape"):
                    print("Shape:", value.shape)
        else:
            if hasattr(measurement, "shape"):
                print("Shape:", measurement.shape)
            else:
                print(measurement)

        results.append({"Classes": k,
                        "LogLik": loglik,
                        "AIC": model.aic(X),
                        "BIC": model.bic(X),
                        "Converged": model.converged_,
                        "Iterations": model.n_iter_,
                        "MeanAssignment": assignment.mean(),
                        "MedianAssignment": np.median(assignment),
                        "MinClass": proportions.min(),
                        "MaxClass": proportions.max(),
                        "EffectiveClasses": (proportions > 0.01).sum()})
        
    except Exception as e:
        results.append({"Classes": k, "Error": str(e)})

    pd.DataFrame(results).to_csv("lca_candidate_models_results.csv", index=False)

Fitting 24 classes...
Fitting StepMix...


Initializations (n_init) :   0%|          | 0/50 [00:00<?, ?it/s]

Initializations (n_init) : 100%|██████████| 50/50 [37:35<00:00, 45.11s/it, max_LL=-1.15e+6, max_avg_LL=-13.5]


Log-likelihood : -1,154,026.29
AIC            : 2,310,978.58
BIC            : 2,324,669.52
Converged      : True
Iterations     : 228

Class proportions
0     0.0270
1     0.0183
2     0.1043
3     0.0135
4     0.0891
5     0.0222
6     0.0314
7     0.0823
8     0.0369
9     0.0560
10    0.0839
11    0.0327
12    0.0725
13    0.0350
14    0.0063
15    0.0175
16    0.0210
17    0.0362
18    0.0282
19    0.0192
20    0.0225
21    0.0390
22    0.0866
23    0.0186
dtype: float64

Observed class sizes
0     0.0251
1     0.0180
2     0.1066
3     0.0131
4     0.0949
5     0.0174
6     0.0300
7     0.0914
8     0.0346
9     0.0556
10    0.0903
11    0.0335
12    0.0757
13    0.0333
14    0.0059
15    0.0156
16    0.0207
17    0.0342
18    0.0264
19    0.0187
20    0.0184
21    0.0409
22    0.0806
23    0.0191
Name: proportion, dtype: float64

Posterior assignment certainty
count    85659.000000
mean         0.767623
std          0.197559
min          0.219617
25%          0.593647
50%        

Initializations (n_init) : 100%|██████████| 50/50 [36:48<00:00, 44.18s/it, max_LL=-1.15e+6, max_avg_LL=-13.5]


Log-likelihood : -1,152,373.75
AIC            : 2,308,161.49
BIC            : 2,324,135.82
Converged      : True
Iterations     : 281

Class proportions
0     0.0792
1     0.0158
2     0.0526
3     0.0420
4     0.0359
5     0.0192
6     0.0229
7     0.0180
8     0.0261
9     0.0571
10    0.0069
11    0.0275
12    0.0040
13    0.0204
14    0.0485
15    0.0391
16    0.0175
17    0.0689
18    0.0266
19    0.0378
20    0.0326
21    0.0358
22    0.0725
23    0.0594
24    0.0198
25    0.0415
26    0.0564
27    0.0157
dtype: float64

Observed class sizes
0     0.0779
1     0.0133
2     0.0627
3     0.0361
4     0.0341
5     0.0188
6     0.0201
7     0.0182
8     0.0245
9     0.0643
10    0.0064
11    0.0296
12    0.0038
13    0.0196
14    0.0419
15    0.0340
16    0.0170
17    0.0794
18    0.0288
19    0.0370
20    0.0333
21    0.0372
22    0.0715
23    0.0617
24    0.0195
25    0.0379
26    0.0564
27    0.0150
Name: proportion, dtype: float64

Posterior assignment certainty
count    85659.00

In [5]:
results_df = pd.DataFrame(results)

print(results_df)

   Classes        LogLik           AIC           BIC  Converged  Iterations  \
0       24 -1.154026e+06  2.310979e+06  2.324670e+06       True         228   
1       28 -1.152374e+06  2.308161e+06  2.324136e+06       True         281   

   MeanAssignment  MedianAssignment  MinClass  MaxClass  EffectiveClasses  
0        0.767623          0.811638  0.006293   0.10432                23  
1        0.746739          0.770802  0.004037   0.07922                26  
